# CSP with SVM

Run the loading and QC cells to check the trial counts for your recordings. The saved outputs come from earlier exploratory runs on data that informed model development.

In [ ]:
from pathlib import Path
import sys

import mne
import numpy as np
import pandas as pd

from mne.decoding import CSP
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "clean.py").is_file()),
)
sys.path.insert(0, str(PROJECT_ROOT))
from loader import get_files
from clean import load_dataset, save_qc

mne.set_log_level("WARNING")

### Load screened imagery trials

`load_dataset()` applies the fixed channel and trial QC rules in `clean.py`. The trial QC table keeps one row per original trial, including rejection reasons. This cell also updates the reports in `data/processed/qc`.

In [4]:
files = get_files()
dataset = load_dataset(files)
epochs = dataset.epochs
qc_summary = save_qc(dataset, files)

print(f"Kept {len(epochs)} of {len(dataset.trial_qc)} trials")
print(f"Interpolated channel/run pairs: {dataset.channel_qc['interpolated'].sum()}")
print(qc_summary.loc[qc_summary["review_required"]].to_string(index=False))

/Users/trentonpham/Code/eeg-motor-imagery/clean.py:238: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  mne.concatenate_epochs([result.epochs for result in results]),
/Users/trentonpham/Code/eeg-motor-imagery/clean.py:238: RuntimeWarning: One of the Epochs objects to concatenate was empty.
  mne.concatenate_epochs([result.epochs for result in results]),
/Users/trentonpham/Code/eeg-motor-imagery/clean.py:238: RuntimeWarning: One of the Epochs objects to concatenate was empty.
  mne.concatenate_epochs([result.epochs for result in results]),
/Users/trentonpham/Code/eeg-motor-imagery/clean.py:238: RuntimeWarning: One of the Epochs objects to concatenate was empty.
  mne.concatenate_epochs([result.epochs for result in results]),
/Users/trentonpham/Code/eeg-motor-imagery/clean.py:238: RuntimeWarning: One of the Epochs objects to concatenate was empty.
  mne.concatenate_epochs([result.epochs for result in results]),
/Users/tr

Kept 729 of 900 trials
Interpolated channel/run pairs: 4
 subject  run  condition  total  kept  rejected  review_required
       9    4  left_hand      8     0         8             True
       9    4 right_hand      7     0         7             True
       9    8  left_hand      8     0         8             True
       9    8 right_hand      7     0         7             True
       9   12  left_hand      8     0         8             True
       9   12 right_hand      7     0         7             True
      13    4  left_hand      8     0         8             True
      13    4 right_hand      7     0         7             True
      13    8  left_hand      8     0         8             True
      13    8 right_hand      7     0         7             True
      13   12  left_hand      7     0         7             True
      13   12 right_hand      8     0         8             True
      17    8  left_hand      8     0         8             True
      17    8 right_hand      7  

### Prepare model inputs

`X` contains screened EEG in volts, with dimensions for trials, channels, and time samples. `y` contains the integer labels 1 and 2. `subjects` and `runs` identify the groups used to keep validation trials separate from training trials.

In [19]:
X = epochs.get_data(copy=True)
y = epochs.events[:, -1]

subjects = epochs.metadata["subject"].to_numpy()
runs = epochs.metadata["run"].to_numpy()

print("EEG array:", X.shape)
print("Labels:", np.unique(y, return_counts=True))

EEG array: (729, 64, 481)
Labels: (array([1, 2]), array([364, 365]))


### Check class balance

The table below reports left-hand and right-hand trial counts after QC for each subject. The analysis does not oversample either class. Review the flagged run and condition rows from the loading cell before interpreting model results.

In [20]:
trial_table = epochs.metadata.copy()
trial_table["condition"] = np.where(
    y == 1,
    "left_hand",
    "right_hand",
)

pd.crosstab(
    trial_table["subject"],
    trial_table["condition"],
    margins=True,
)

condition,left_hand,right_hand,All
subject,,,
1,22,22,44
2,23,22,45
3,21,20,41
4,23,22,45
5,21,24,45
6,21,19,40
7,23,21,44
8,22,23,45
10,24,20,44


## Evaluate CSP with a linear SVM

In [5]:
rows = []

csp_svm = make_pipeline(
    CSP(reg="ledoit_wolf", log=True, norm_trace=False),
    SVC(kernel="linear", C=1.0, class_weight="balanced")
)

param_grid = {
    "csp__n_components": [2, 4, 6],
}

for train, test in LeaveOneGroupOut().split(X, y, groups=subjects):
    search = GridSearchCV(
        csp_svm,
        param_grid,
        cv=LeaveOneGroupOut(),
        scoring="balanced_accuracy",
        refit=True,
        error_score="raise",
    )
    search.fit(X[train], y[train], groups=subjects[train])

    best_model = search.best_estimator_
    predictions = best_model.predict(X[test])

    rows.append({
        "subject": int(subjects[test][0]),
        "n_components": search.best_params_["csp__n_components"],
        "balanced_accuracy": balanced_accuracy_score(y[test], predictions),
    })

results = pd.DataFrame(rows)
display(results)
print("Mean balanced accuracy:", results["balanced_accuracy"].mean())

,subject,n_components,balanced_accuracy
0,1,6,0.704545
1,2,6,0.660079
2,3,6,0.394048
3,4,6,0.500000
4,5,4,0.491071
5,6,4,0.483709
6,7,6,0.932712
7,8,6,0.578063
8,10,6,0.608333
9,11,6,0.454545


Mean balanced accuracy: 0.5952210070631123
